### RAG with Tabalar Data and Vector Memory


In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName('Chatbot_rag_v2') \
    .config("spark.jars", "/opt/spark/jars/iceberg-spark-runtime-3.5_2.12-1.6.0.jar") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.spark_catalog.type", "hive") \
    .config("spark.sql.catalog.local.warehouse", "s3a://datalake/iceberg") \
    .getOrCreate()

#Ajuste de log WARN log para ERROR
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
!pip install langchain==0.2.17
!pip install langchain_community==0.2.19
!pip install -qU langchain-ollama==0.1.3
!pip install -qU langchain-qdrant==0.1.4

In [ ]:
import os
from dotenv import load_dotenv


### Visualizar e pegar uma amostra dos dados

In [ ]:
spark.sql("Select * from iceberg.silver.tbl_silver_olhovivo").limit(10).show()

In [ ]:
df = spark.sql("""
    Select
    c,
    cl,
    sl,
    lt0,
    lt1,
    qv
    
    from iceberg.silver.tbl_silver_olhovivo """
).limit(10)

df.createOrReplaceTempView("vw_silver_olhovivo")

spark.sql("select * from vw_silver_olhovivo").show()

## Funções Auxiliares

In [ ]:
import re

def limpar_sql(resposta_modelo):
    """Remove blocos de código markdown"""
    sql = re.sub(r"```sql|```", "", resposta_modelo, flags=re.IGNORECASE).strip()
    return sql

In [ ]:
def get_metadata(table_name):
    df = spark.sql(f"SELECT * FROM {table_name} LIMIT 1;")
    colunas = "\n".join([f"- {f.name}: {f.dataType.simpleString()}" for f in df.schema])
    return f"Tabela: {table_name}\n\nColunas:\n{colunas}"


## Qdrant Memory

In [ ]:
load_dotenv('../.env')
OLLAMA_API_URL = os.getenv("OLLAMA_API_URL")


In [ ]:
from langchain_ollama import OllamaEmbeddings

embedding = OllamaEmbeddings(model="mistral:latest", base_url=OLLAMA_API_URL)

In [ ]:
# Cria coleção para armazenar os embeddings 

from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue
from langchain_core.documents import Document
import uuid

client = QdrantClient(":memory:")

collection_name ="olho_vivo"

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=4096, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embedding,
)

retriever = vector_store.as_retriever(search_kwargs={"k": 2})

In [ ]:
%run ./Memory.ipynb

In [ ]:
qdrant_memory = QdrantMemory(client, embedding)

## Iniciar Mistral 7B

In [ ]:
from langchain_ollama import ChatOllama

# llm = ChatOllama(
#     model="mistral:latest", 
#     base_url=OLLAMA_API_URL,
#     temperature = 0.3,
 
# )

llm = ChatOllama(
    model="mistral:latest", 
    base_url=OLLAMA_API_URL,
    temperature=0.3,
    num_predict=200,
    top_k=30,
    top_p=0.9,
    repeat_penalty=1.1
    
) 

### Configurar Promps: Roles System e Human

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

# Prompt para gerar SQL (Spark)

prompt_sql = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("Você é um especialista em dados. Gere apenas a consulta SQL."),
    HumanMessagePromptTemplate.from_template(
        "Com base na estrutura da tabela abaixo:\n\n{schema}\n\n"
        "Escreva uma consulta SQL (somente a SQL) para responder:\n{pergunta}"
    )
])


# Prompt para retornar resultado ao usuario

prompt_resposta = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("Você é um assistente de dados."),
    HumanMessagePromptTemplate.from_template(
        "Pergunta: {pergunta}\n\nResultado da consulta:\n{resultado}\n\n"
        "Gere uma resposta clara e amigável para o usuário contendo apenas os resultados da consulta."
    )
])


In [ ]:
# Função para gerar resposta com RAG (Tabela + Qdrant)
def augmented_response(pergunta, table_name):
    print("\n💬 Pergunta recebida:", pergunta)

    #Obter metadados da tabela
    schema_txt = get_metadata(table_name)

    # Busca embeddings no Qdrant (Memoria)
    docs = retriever.invoke(pergunta)
    contexto = "\n".join([doc.page_content for doc in docs])

    # Gerar o SQL da query com base na pergunta
    sql_chain = prompt_sql | llm
    sql_result = sql_chain.invoke({
        "pergunta": pergunta, 
        "schema": schema_txt, 
        "context": contexto
    }).content.strip()
    
    # sql_result = llm.invoke(prompt_sql.format_messages(pergunta=pergunta, schema=schema_txt, context=contexto)).content.strip() prompt formatado
    sql_query=limpar_sql(sql_result)
    print("\n🤖💡 SQL Gerada:", sql_query)

    # Executa query no Spark
    try:

        resultado_df = spark.sql(sql_query).toPandas().to_dict(orient="records")        
    except Exception as e:
        print("\n❌ Erro na execução da SQL:", e)
        return

    # Gera resposta amigável para retornar ao usuario
    resposta_chain = prompt_resposta | llm
    resposta = resposta_chain.invoke({
        "pergunta": pergunta, 
        "resultado": resultado_df
    }).content.strip()
    # resposta = llm.invoke(prompt_resposta.format_messages(pergunta=pergunta, resultado=resultado_dict)).content.strip() prompt formatado
    print("\n🤖 Resposta final:", resposta)

    # Armazena embeddings da pergunta + SQL no Qdrant
    qdrant_memory.ensinar(pergunta, sql_query, metadados={"tabela": table_name, "resultado": resultado_df, "schema": schema_txt, "score": 1})
    

    return resposta

In [ ]:
resp = augmented_response("Qual a origem e destinho do viculo 746K-10 (c=746K-10), me fala a direção atual dele?", 
                        "vw_silver_olhovivo"
)

### Listar Exemplos de Consultas

In [ ]:
qdrant_memory.listar_exemplos()

In [ ]:
qdrant_memory.list_scored_point("Qual a origem (lt0) e destinho (lt1) do viculo 746K-10, me fala a direção atual dele (sl, 2 destino. 1 origem)?")

## "Ensinar" refinar comportamento manualmente

In [ ]:
sql_query =""" SELECT lt0, lt1, sl FROM vw_silver_olhovivo WHERE c = '746K-10'"""

In [ ]:
spark.sql(sql_query).show()

In [ ]:
table_name = "vw_silver_olhovivo"
schema_txt = get_metadata(table_name)

qdrant_memory.ensinar(
    "Qual a origem (lt0) e destinho(lt1) do veiculo 746k-10, me fala a direção atual dele (sl, 2 destino. 1 origem)?",
    sql_query, 
    metadados={"tabela": table_name, "tipo": "AGG", "schema": schema_txt, "score": 1}
)